# Interactive Tensor-Parallel Chat

Select your normal **Python 3** kernel and set **Processes: 2** before running. This notebook loads Qwen with Transformers native tensor parallelism, sends one prompt, and then continues the same conversation with the live model.

The demo showcases how a distributed model can retain the familiar notebook workflow: run it, inspect it, and continue working with it just as you would a non-distributed model. Qwen3-0.6B keeps the example approachable, but try replacing it with the largest model that fits on your machine.

> This notebook assumes your environment contains the following dependencies (apart from the jupyter-distributed extension):
> - torch
> - transformers >= 5.15
> - accelerate
> - tqdm
> - ipywidgets

Initialize PyTorch distributed and select the CUDA device assigned to this process by jupyter-distributed.

In [ ]:
import os

import torch
import torch.distributed as dist

local_rank = int(os.environ["LOCAL_RANK"])
torch.cuda.set_device(local_rank)
device = torch.device("cuda", local_rank)
if not dist.is_initialized():
    dist.init_process_group("nccl")

Load Qwen with Transformers' built-in tensor-parallel plan. Every process participates in the same model calls; Transformers uses the existing distributed process group instead of launching another job.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.distributed import DistributedConfig

model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
distributed_config = DistributedConfig(tp_size=dist.get_world_size())
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    distributed_config=distributed_config,
)
model.eval()

Inspect one sharded projection. Transformers stores the weight as an ordinary rank-local tensor and attaches tensor-parallel communication hooks to its module. Compare the full checkpoint shape with the local shard loaded by this process.

In [ ]:
projection = model.model.layers[0].self_attn.q_proj
{
    "parameter": "model.layers.0.self_attn.q_proj.weight",
    "tp_plan": model.config.base_model_tp_plan["layers.*.self_attn.q_proj"],
    "global_shape": (
        model.config.num_attention_heads * model.config.head_dim,
        model.config.hidden_size,
    ),
    "local_shape": tuple(projection.weight.shape),
}

Define a small generation helper and start the conversation with one prompt. Every rank enters generation and displays its result.

In [ ]:
def generate(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=1024, do_sample=False)
    answer = tokenizer.decode(
        output[0, inputs.input_ids.shape[1] :],
        skip_special_tokens=True,
    )
    messages.append({"role": "assistant", "content": answer})
    return answer


messages = [{"role": "user", "content": "In two sentences, what is tensor parallelism?"}]
print(generate(messages))

Edit `prompt` and rerun the next cell to continue the conversation and display the full chat.

In [ ]:
def print_chat(messages):
    for message in messages:
        print(f"{message['role'].title()}: {message['content']}\n")


prompt = "Give a two-line pseudocode example of an all-reduce."
messages.append({"role": "user", "content": prompt})
generate(messages)
print_chat(messages)